Using the BIS data portal which keeps an updated version of all central bank speeches, I remove all non relevant ones and retain simply all the Banque de france speechs and save it in a respective folder, all ecb speeches and save it in a respective folder, and finally all FED speechs and save it in a respective folder. 

Packages

In [2]:
from pathlib import Path
import pandas as pd
import numpy as np

Path and original dataframe

In [ ]:
central_bank_speeches_path = Path("/Users/lorenzouberti/Desktop/BANQUE DE FRANCE /Data Folders/")
central_bank_speeches = pd.read_csv(central_bank_speeches_path / "speeches.csv")

Index(['url', 'title', 'description', 'date', 'text', 'author'], dtype='object')
(20365, 6)


In [5]:
print(central_bank_speeches.columns)
print(central_bank_speeches.shape)

Index(['url', 'title', 'description', 'date', 'text', 'author'], dtype='object')
(20365, 6)


In [12]:
print(central_bank_speeches['description'].dropna().unique())

["Speech by the Deputy Governor of the People's Bank of China, Mr. Chen Yuan, at the Bank of England Seminar held in London on 10/9/96."
 "Speech by the Governor of the People's Bank of China, Mr. Dai Xianglong, at the Hong Kong Association of Banks' Dinner on 13/11/96."
 "Speech by the Governor of the People's Bank of China, Mr. Dai Xianglong, at the luncheon hosted by the Hong Kong Financial Secretary in Washington on 30/9/96."
 ...
 'Speech by Mr José Luis Escrivá, Governor of the Bank of Spain, at the Bank of Spain seminar "SME financing: new outlooks for securitisations", Madrid, 31 October 2025.'
 'Speech by Mr Kazuo Ueda, Governor of the Bank of Japan, at the Meeting of Councillors of Keidanren (Japan Business Federation), Tokyo, 25 December 2025.'
 'Text of the 10th Bundesbank IAW Lecture by Dr Joachim Nagel, President of the Deutsche Bundesbank, at the Institute for Applied Economic Research (IAW), University of Tübingen, Tübingen, 9 July 2027.']


In [13]:
# ...existing code...
import re
from pathlib import Path

# Show columns to confirm text fields
print("columns:", central_bank_speeches.columns.tolist())

# text columns to search
text_cols = central_bank_speeches.select_dtypes(include='object').columns.tolist()
print("text columns searched:", text_cols)

def rows_for_keywords(df, keywords):
    pattern = '|'.join(re.escape(k) for k in keywords)
    mask = df[text_cols].fillna('').apply(lambda col: col.str.contains(pattern, case=False, regex=True)).any(axis=1)
    return df[mask]

# keyword lists (adjust if you see misses)
fed_keywords = ['federal reserve', r'\bFED\b', 'fed', 'federal reserve bank', 'board of governors']
ecb_keywords = ['european central bank', r'\bECB\b', 'european central bank']
bdf_keywords = ['banque de france', 'banque de france']

fed = rows_for_keywords(central_bank_speeches, fed_keywords)
ecb = rows_for_keywords(central_bank_speeches, ecb_keywords)
bdf = rows_for_keywords(central_bank_speeches, bdf_keywords)

# Quick checks
print("FED hits:", fed.shape[0])
print("ECB hits:", ecb.shape[0])
print("Banque de France hits:", bdf.shape[0])

# Show a few title/description samples to validate
for name, df in [('FED', fed), ('ECB', ecb), ('Banque de France', bdf)]:
    print(name, "sample descriptions:")
    if 'description' in df.columns:
        print(df['description'].dropna().unique()[:5])
    else:
        print(df.head(3))

# Save filtered files
out_dir = Path("/Users/lorenzouberti/Desktop/BANQUE DE FRANCE /Data Folders/filtered_speeches")
out_dir.mkdir(parents=True, exist_ok=True)
fed.to_csv(out_dir / "fed_speeches.csv", index=False)
ecb.to_csv(out_dir / "ecb_speeches.csv", index=False)
bdf.to_csv(out_dir / "banque_de_france_speeches.csv", index=False)

columns: ['url', 'title', 'description', 'date', 'text', 'author']
text columns searched: ['url', 'title', 'description', 'date', 'text', 'author']
FED hits: 8010
ECB hits: 4310
Banque de France hits: 594
FED sample descriptions:
['Speech by the Deputy Governor of the Bank of England, Mr. Howard Davies, at the EFMA Conference on the Single Currency in Brussels on 16/12/96.'
 'Remarks by the Vice Chairman of the Board of Governors of the US Federal Reserve System, Ms. Alice M. Rivlin, at the The Brookings Institution National Issues Forum in Washington on 19/12/96.'
 'Translation of an article by the President of the Deutsche Bundesbank, Prof. Hans Tietmeyer, published in the Frankfurter Allgemeine Zeitung on 28/12/96.'
 'Remarks by Mr. Laurence H. Meyer, a member of the Board of Governors of the US Federal Reserve System, at the Forecasters Club of New York on 24/4/97.'
 'Address by the Deputy Governor of the Bank of Sweden, Mr. Lars Heikensten, at the Monetary Policy Forum held in Sto